In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum, avg, count, desc, upper
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]   
df=spark.createDataFrame(data, schema=columns)


In [0]:
df = df.withColumn("test_cost", col("tests_count") * 2000) \
       .withColumn("total_bill", col("consultation_fee") + col("test_cost")) \
       .withColumn("patient_category",
           when(col("total_bill") >= 9000, "Premium")
          .when(col("total_bill") >= 5000, "Standard")
          .otherwise("Basic")
       ) \
       .withColumn("department", upper(col("department")))

df.show()


+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad| CARDIOLOGY|            5000|          1|     2000|      7000|        Standard|
|     102|Sneha Kapoor|    Delhi|ORTHOPEDICS|            3000|          2|     4000|      7000|        Standard|
|     103|Rahul Sharma|   Mumbai|DERMATOLOGY|            1500|          1|     2000|      3500|           Basic|
|     104|  Priya Nair|Bangalore| CARDIOLOGY|            5000|          2|     4000|      9000|         Premium|
|     105|Vikram Singh|  Chennai|  NEUROLOGY|            7000|          1|     2000|      9000|         Premium|
|     106|  Ananya Das|  Kolkata|ORTHOPEDICS|            3000|          3|     6000|      9000| 

In [0]:
high_value = df.filter(col("total_bill") >= 9000)
high_value.show()

+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|     104|  Priya Nair|Bangalore| CARDIOLOGY|            5000|          2|     4000|      9000|         Premium|
|     105|Vikram Singh|  Chennai|  NEUROLOGY|            7000|          1|     2000|      9000|         Premium|
|     106|  Ananya Das|  Kolkata|ORTHOPEDICS|            3000|          3|     6000|      9000|         Premium|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+



In [0]:
dept_summary = df.groupBy("department") \
    .agg(
        count("visit_id").alias("total_patients"),
        sum("consultation_fee").alias("total_fee"),
        avg("total_bill").alias("avg_bill")
    )

dept_summary.show()

+-----------+--------------+---------+-----------------+
| department|total_patients|total_fee|         avg_bill|
+-----------+--------------+---------+-----------------+
| CARDIOLOGY|             3|    15000|7666.666666666667|
|ORTHOPEDICS|             2|     6000|           8000.0|
|DERMATOLOGY|             2|     3000|           4500.0|
|  NEUROLOGY|             1|     7000|           9000.0|
+-----------+--------------+---------+-----------------+



In [0]:
dept_summary.orderBy(desc("total_fee")).show()

+-----------+--------------+---------+-----------------+
| department|total_patients|total_fee|         avg_bill|
+-----------+--------------+---------+-----------------+
| CARDIOLOGY|             3|    15000|7666.666666666667|
|  NEUROLOGY|             1|     7000|           9000.0|
|ORTHOPEDICS|             2|     6000|           8000.0|
|DERMATOLOGY|             2|     3000|           4500.0|
+-----------+--------------+---------+-----------------+



# PART 2

In [0]:
df.createOrReplaceTempView("hospital_visits")

In [0]:

spark.sql("""
  SELECT visit_id, patient_name, city, consultation_fee
  FROM hospital_visits
  WHERE department = 'CARDIOLOGY'
""").show()
spark.sql("""
  SELECT city,
         SUM(consultation_fee) AS total_fee,
         SUM(total_bill)       AS total_revenue
  FROM hospital_visits
  GROUP BY city
  ORDER BY total_revenue DESC
""").show()

+--------+------------+---------+----------------+
|visit_id|patient_name|     city|consultation_fee|
+--------+------------+---------+----------------+
|     101| Arjun Reddy|Hyderabad|            5000|
|     104|  Priya Nair|Bangalore|            5000|
|     107| Karan Patel|Ahmedabad|            5000|
+--------+------------+---------+----------------+

+---------+---------+-------------+
|     city|total_fee|total_revenue|
+---------+---------+-------------+
|Bangalore|     6500|        14500|
|  Chennai|     7000|         9000|
|  Kolkata|     3000|         9000|
|Hyderabad|     5000|         7000|
|    Delhi|     3000|         7000|
|Ahmedabad|     5000|         7000|
|   Mumbai|     1500|         3500|
+---------+---------+-------------+



In [0]:
spark.sql("""
  SELECT patient_name, department, total_bill
  FROM hospital_visits
  ORDER BY total_bill DESC
  LIMIT 3
""").show()

spark.sql("""
  SELECT department,
         COUNT(*) AS patient_count,
         AVG(consultation_fee) AS avg_fee
  FROM hospital_visits
  GROUP BY department
  ORDER BY patient_count DESC
""").show()

+------------+-----------+----------+
|patient_name| department|total_bill|
+------------+-----------+----------+
|  Priya Nair| CARDIOLOGY|      9000|
|Vikram Singh|  NEUROLOGY|      9000|
|  Ananya Das|ORTHOPEDICS|      9000|
+------------+-----------+----------+

+-----------+-------------+-------+
| department|patient_count|avg_fee|
+-----------+-------------+-------+
| CARDIOLOGY|            3| 5000.0|
|ORTHOPEDICS|            2| 3000.0|
|DERMATOLOGY|            2| 1500.0|
|  NEUROLOGY|            1| 7000.0|
+-----------+-------------+-------+



# PART-3

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("hospital_delta")

spark.sql("SELECT * FROM hospital_delta").show()
spark.sql("""
  INSERT INTO hospital_delta VALUES
  (109, 'Ravi Kumar', 'Pune', 'NEUROLOGY', 7000, 2, 4000, 11000, 'Premium'),
  (110, 'Divya Menon', 'Chennai', 'CARDIOLOGY', 5000, 1, 2000, 7000, 'Standard')
""")

+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad| CARDIOLOGY|            5000|          1|     2000|      7000|        Standard|
|     102|Sneha Kapoor|    Delhi|ORTHOPEDICS|            3000|          2|     4000|      7000|        Standard|
|     103|Rahul Sharma|   Mumbai|DERMATOLOGY|            1500|          1|     2000|      3500|           Basic|
|     104|  Priya Nair|Bangalore| CARDIOLOGY|            5000|          2|     4000|      9000|         Premium|
|     105|Vikram Singh|  Chennai|  NEUROLOGY|            7000|          1|     2000|      9000|         Premium|
|     106|  Ananya Das|  Kolkata|ORTHOPEDICS|            3000|          3|     6000|      9000| 

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
  UPDATE hospital_delta
  SET consultation_fee = 6000,
      total_bill = 6000 + (tests_count * 2000)
  WHERE department = 'NEUROLOGY'
""")
spark.sql("""
  DELETE FROM hospital_delta
  WHERE consultation_fee < 2000
""")


DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql("""
  CREATE OR REPLACE TEMP VIEW incoming_updates AS
  SELECT * FROM VALUES
    (104, 'Priya Nair', 'Bangalore', 'CARDIOLOGY', 6000, 3, 6000, 12000, 'Premium'),
    (111, 'Suresh Babu', 'Hyderabad', 'ORTHOPEDICS', 3000, 2, 4000, 7000, 'Standard')
  AS t(visit_id, patient_name, city, department,
       consultation_fee, tests_count, test_cost, total_bill, patient_category)
""")

spark.sql("""
  MERGE INTO hospital_delta AS target
  USING incoming_updates AS source
  ON target.visit_id = source.visit_id
  WHEN MATCHED THEN
    UPDATE SET *
  WHEN NOT MATCHED THEN
    INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

# PART-4

In [0]:
spark.sql("DESCRIBE HISTORY hospital_delta").show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
spark.sql("""
  SELECT * FROM hospital_delta VERSION AS OF 0
""").show()

# By timestamp
spark.sql("""
  SELECT * FROM hospital_delta
  TIMESTAMP AS OF '2026-05-04 04:16:41'
""").show()

# PySpark way
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("hospital_delta")
df_v0.show()

+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad| CARDIOLOGY|            5000|          1|     2000|      7000|        Standard|
|     102|Sneha Kapoor|    Delhi|ORTHOPEDICS|            3000|          2|     4000|      7000|        Standard|
|     103|Rahul Sharma|   Mumbai|DERMATOLOGY|            1500|          1|     2000|      3500|           Basic|
|     104|  Priya Nair|Bangalore| CARDIOLOGY|            5000|          2|     4000|      9000|         Premium|
|     105|Vikram Singh|  Chennai|  NEUROLOGY|            7000|          1|     2000|      9000|         Premium|
|     106|  Ananya Das|  Kolkata|ORTHOPEDICS|            3000|          3|     6000|      9000| 

In [0]:
spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled = false")
spark.sql("VACUUM hospital_delta RETAIN 168 HOURS DRY RUN").show()

# Part - 5

In [0]:

# Save as parquet
df.write.format("parquet") \
  .mode("overwrite") \
  .save("dbfs:/Volumes/hospital_catalog/hospital_db/hospital_vol/hospital_parquet")

# Verify
display(dbutils.fs.ls("dbfs:/Volumes/hospital_catalog/hospital_db/hospital_vol/hospital_parquet"))parquet_df = spark.read.format("parquet").load("/tmp/hospital_parquet")

parquet_df.write.format("delta") \
  .mode("overwrite") \
  .save("/tmp/hospital_converted_delta")

# Method 2: Convert in-place using SQL (faster for large tables)
spark.sql("""
  CONVERT TO DELTA parquet.`/tmp/hospital_parquet`
""")
converted_df = spark.read.format("delta").load("/tmp/hospital_converted_delta")
converted_df.show()

# Verify row count matches original
print(f"Original rows: {df.count()}")
print(f"Converted rows: {converted_df.count()}")

# Check Delta transaction log exists
display(dbutils.fs.ls("/tmp/hospital_converted_delta/_delta_log"))

# Part - 6

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("hospital_target")

spark.sql("SELECT COUNT(*) as initial_count FROM hospital_target").show()

+-------------+
|initial_count|
+-------------+
|            8|
+-------------+



In [0]:
daily_data = [
    (104, "Priya Nair", "Bangalore", "CARDIOLOGY", 6500, 3, 6000, 12500, "Premium"),
    (108, "Meera Iyer", "Bangalore", "DERMATOLOGY", 2000, 2, 4000, 6000, "Standard"),
    (112, "Arun Kumar", "Pune", "NEUROLOGY", 7000, 2, 4000, 11000, "Premium"),
    (113, "Kavya Reddy", "Delhi", "ORTHOPEDICS", 3000, 1, 2000, 5000, "Standard")
]

daily_df = spark.createDataFrame(daily_data, columns + ["test_cost","total_bill","patient_category"])
daily_df.createOrReplaceTempView("daily_updates")

In [0]:
spark.sql("""
  MERGE INTO hospital_target AS target
  USING daily_updates AS source
  ON target.visit_id = source.visit_id

  WHEN MATCHED THEN
    UPDATE SET
      target.consultation_fee = source.consultation_fee,
      target.tests_count      = source.tests_count,
      target.test_cost        = source.test_cost,
      target.total_bill       = source.total_bill,
      target.patient_category = source.patient_category

  WHEN NOT MATCHED THEN
    INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT COUNT(*) as final_count FROM hospital_target").show()
spark.sql("SELECT * FROM hospital_target ORDER BY visit_id").show()
spark.sql("DESCRIBE HISTORY hospital_target").show()

+-----------+
|final_count|
+-----------+
|         10|
+-----------+

+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad| CARDIOLOGY|            5000|          1|     2000|      7000|        Standard|
|     102|Sneha Kapoor|    Delhi|ORTHOPEDICS|            3000|          2|     4000|      7000|        Standard|
|     103|Rahul Sharma|   Mumbai|DERMATOLOGY|            1500|          1|     2000|      3500|           Basic|
|     104|  Priya Nair|Bangalore| CARDIOLOGY|            6500|          3|     6000|     12500|         Premium|
|     105|Vikram Singh|  Chennai|  NEUROLOGY|            7000|          1|     2000|      9000|         Premium|
|     106|  Ananya Das|  

# Parrt - 7

In [0]:
import dlt
from pyspark.sql.functions import col, upper, when

@dlt.table(
    name="bronze_hospital_visits",
    comment="Raw hospital visits data — no transformations"
)
def bronze_hospital_visits():
    data = [
        (101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
        (102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
        (103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
        (104,"Priya Nair","Bangalore","Cardiology",5000,2),
        (105,"Vikram Singh","Chennai","Neurology",7000,1),
        (106,"Ananya Das","Kolkata","Orthopedics",3000,3),
        (107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
        (108,"Meera Iyer","Bangalore","Dermatology",1500,2)
    ]
    columns = ["visit_id","patient_name","city","department",
               "consultation_fee","tests_count"]
    return spark.createDataFrame(data, columns)
@dlt.table(
    name="silver_hospital_visits",
    comment="Cleaned data with derived columns, filtered valid records"
)
def silver_hospital_visits():
    return (
        dlt.read("bronze_hospital_visits")
        .filter(col("consultation_fee") > 0)
        .withColumn("department", upper(col("department")))
        .withColumn("test_cost", col("tests_count") * 2000)
        .withColumn("total_bill", col("consultation_fee") + col("test_cost"))
        .withColumn("patient_category",
            when(col("total_bill") >= 9000, "Premium")
           .when(col("total_bill") >= 5000, "Standard")
           .otherwise("Basic")
        )
    )
@dlt.table(
    name="gold_hospital_summary",
    comment="Department + city level aggregations for reporting"
)
def gold_hospital_summary():
    from pyspark.sql.functions import count, sum as _sum
    return (
        dlt.read("silver_hospital_visits")
        .groupBy("city", "department")
        .agg(
            count("visit_id").alias("total_patients"),
            _sum("tests_count").alias("total_tests"),
            _sum("total_bill").alias("total_revenue")
        )
    )

# part - 8

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS hospital_catalog")
spark.sql("USE CATALOG hospital_catalog")
spark.sql("SHOW CATALOGS").show()
spark.sql("CREATE SCHEMA IF NOT EXISTS hospital_catalog.hospital_db")
spark.sql("USE SCHEMA hospital_catalog.hospital_db")
spark.sql("SHOW SCHEMAS IN hospital_catalog").show()

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-8805255353795717>, line 2
      1 spark.sql("CREATE CATALOG IF NOT EXISTS hospital_catalog")
----> 2 spark.sql("USE CATALOG hospital_catalog")
      3 spark.sql("SHOW CATALOGS").show()
      4 spark.sql("CREATE SCHEMA IF NOT EXISTS hospital_catalog.hospital_db")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, i

In [0]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS hospital_catalog.hospital_db.patient_visits (
    visit_id         INT,
    patient_name     STRING,
    city             STRING,
    department       STRING,
    consultation_fee INT,
    tests_count      INT,
    test_cost        INT,
    total_bill       INT,
    patient_category STRING
  )
  USING DELTA
""")

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("hospital_catalog.hospital_db.patient_visits")

# Verify
spark.sql("SELECT * FROM hospital_catalog.hospital_db.patient_visits").show()
spark.sql("DESCRIBE TABLE hospital_catalog.hospital_db.patient_visits").show()

# PART_9

In [0]:

spark.sql("SHOW TABLES IN hospital_catalog.hospital_db").show()
spark.sql("DESCRIBE EXTENDED hospital_catalog.hospital_db.patient_visits").show(truncate=False)
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN hospital_catalog").show()

In [0]:
spark.sql("""
  CREATE OR REPLACE VIEW hospital_catalog.hospital_db.premium_patients AS
  SELECT visit_id, patient_name, city, department, total_bill
  FROM hospital_catalog.hospital_db.patient_visits
  WHERE patient_category = 'Premium'
""")

spark.sql("SELECT * FROM hospital_catalog.hospital_db.premium_patients").show()

In [0]:
spark.sql("""
  DESCRIBE DETAIL hospital_catalog.hospital_db.patient_visits
""").show(truncate=False)

In [0]:
spark.sql("""
  GRANT SELECT ON TABLE hospital_catalog.hospital_db.patient_visits
  TO `user@example.com`
""")

# Grant to a group
spark.sql("""
  GRANT SELECT ON SCHEMA hospital_catalog.hospital_db
  TO `data_analysts`
""")

# Revoke access
spark.sql("""
  REVOKE SELECT ON TABLE hospital_catalog.hospital_db.patient_visits
  FROM `user@example.com`
""")

# Check permissions
spark.sql("SHOW GRANTS ON TABLE hospital_catalog.hospital_db.patient_visits").show()
spark.sql("""
  SELECT event_time, user_name, action_name, request_params
  FROM system.access.audit
  WHERE service_name = 'databricks'
    AND event_date >= current_date() - INTERVAL 1 DAY
  ORDER BY event_time DESC
  LIMIT 20
""").show(truncate=False)

#Capstone

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum, avg, count, desc, upper

spark = SparkSession.builder.appName("HospitalCapstone").getOrCreate()

data = [
    (101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
    (102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
    (103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
    (104,"Priya Nair","Bangalore","Cardiology",5000,2),
    (105,"Vikram Singh","Chennai","Neurology",7000,1),
    (106,"Ananya Das","Kolkata","Orthopedics",3000,3),
    (107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
    (108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = ["visit_id","patient_name","city","department","consultation_fee","tests_count"]

raw_df = spark.createDataFrame(data, columns)

# Save as Delta — Bronze layer
raw_df.write.format("delta").mode("overwrite").saveAsTable("capstone_bronze")
print("Bronze layer created")
bronze_df = spark.table("capstone_bronze")

silver_df = bronze_df \
    .filter(col("consultation_fee") > 0) \
    .withColumn("department", upper(col("department"))) \
    .withColumn("test_cost", col("tests_count") * 2000) \
    .withColumn("total_bill", col("consultation_fee") + col("test_cost")) \
    .withColumn("patient_category",
        when(col("total_bill") >= 9000, "Premium")
       .when(col("total_bill") >= 5000, "Standard")
       .otherwise("Basic")
    )

silver_df.write.format("delta").mode("overwrite").saveAsTable("capstone_silver")
print("Silver layer created")
silver_df = spark.table("capstone_silver")

gold_df = silver_df \
    .groupBy("city", "department") \
    .agg(
        count("visit_id").alias("total_patients"),
        sum("tests_count").alias("total_tests"),
        sum("total_bill").alias("total_revenue"),
        avg("consultation_fee").alias("avg_fee")
    ) \
    .orderBy(desc("total_revenue"))

gold_df.write.format("delta").mode("overwrite").saveAsTable("capstone_gold")
gold_df.show()
print("Gold layer created")
new_data = [
    (109, "Ravi Kumar", "Pune", "NEUROLOGY", 7000, 2),
    (110, "Divya Menon", "Chennai", "CARDIOLOGY", 5000, 1)
]
new_df = spark.createDataFrame(new_data, columns)
new_df.createOrReplaceTempView("new_arrivals")

spark.sql("""
  MERGE INTO capstone_bronze AS target
  USING new_arrivals AS source
  ON target.visit_id = source.visit_id
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")
print("Incremental load complete")


# 